In [1]:
import os
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import nltk
from nltk import bigrams
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.util import ngrams
from nltk.probability import FreqDist, ConditionalFreqDist, LaplaceProbDist

Load the data sets

In [2]:
# Run once if not already downloaded
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\brock\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\brock\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [3]:
DATA_DIR = os.path.join(os.pardir, '175Project')

COL_NAMES = ['character', 'browsing_page_url', 'word_url', 'word', 'definition', 'sentence']

def load_urban_dataset():
    file_paths = []
    for root, dirs, files in os.walk(os.path.join(DATA_DIR, 'Urban')):
        for f in files:
            if f.endswith('.csv') and f.startswith('urban_data'):
                file_paths.append(os.path.join(root, f))
    df_urban = pd.concat([pd.read_csv(f, names=COL_NAMES) for f in file_paths])

    df_nulls = df_urban[(df_urban.isnull().any(axis=1)) | (df_urban.isna().any(axis=1))]
    df_urban = df_urban.drop(df_nulls.index)

    return df_urban

In [4]:
urban_dictionary = load_urban_dataset()
print(f"Shape of urban dictionary dataset: {urban_dictionary.shape}")
ud_sample = urban_dictionary[['word', 'definition', 'sentence']].sample(1)
for i in ud_sample.values:
    print("Word: ", i[0])
    print("Meaning: ", i[1])
    print("Sentence: ", i[2])

Shape of urban dictionary dataset: (2175494, 6)
Word:  oliver's rule
Meaning:  Oliver's Rule  is  called out   prior to  a friend putting their foot in their mouth. 
Sentence:  Last night while clubbing, I shouted " Oliver's Rule !", before  my hoes  comments got her  ass up  into some shit. 


In [5]:
urban_data = urban_dictionary[['word', 'definition', 'sentence']]
train_u, test_u = train_test_split(urban_data, test_size=0.2, random_state=42, shuffle=True)
#example of what the data looks like
row = train_u.iloc[0]
print(row)
print()
print("The full item")
print()
print(row.values)

word                                                     Adeogo
definition    A beautiful tall black girl who's future job i...
sentence      Adeogo has been  playing the violin   for 7   ...
Name: 17480, dtype: object

The full item

['Adeogo'
 "A beautiful tall black girl who's future job is a fashion model for high and popular brands, like Gucci, Dior,  MCM , Louis Vuitton,  Balenciaga  and more. Her horoscope sign is cancer. She could  play the violin  really well. She could also be annoying sometimes, but she's smart and always loves to watch movies and is always kind and loves to spend time with family and friends."
 'Adeogo has been  playing the violin   for 7   years  since she was 6.']


In [25]:
from convokit import Corpus, download
corpus = Corpus(filename=download("movie-corpus"))
corpus.print_summary_stats()

Number of Speakers: 9035
Number of Utterances: 304713
Number of Conversations: 83097


In [26]:
corpus.print_summary_stats()

Number of Speakers: 9035
Number of Utterances: 304713
Number of Conversations: 83097


In [27]:
conversations_raw = []

# Iterate over all conversation IDs
for convo_id in corpus.get_conversation_ids():
    convo = corpus.get_conversation(convo_id)
    # Extract the textual content of each utterance in the conversation
    convo_text = [utt.text for utt in convo.iter_utterances()]
    conversations_raw.append(convo_text)

print(f"Total conversations: {len(conversations_raw)}")
print("Example conversation:", conversations_raw[0])

Total conversations: 83097
Example conversation: ['They do not!', 'They do to!']


In [28]:
pairs = []
for convo in conversations_raw:
    for i in range(len(convo)-1):
        pairs.append((convo[i], convo[i+1]))

print(f"Total pairs: {len(pairs)}")
print("Sample pair:", pairs[0])

Total pairs: 221616
Sample pair: ('They do not!', 'They do to!')


In [29]:
train_convos, test_convos = train_test_split(conversations_raw, test_size=0.2, random_state=42)

In [30]:
all_texts = [utt.text for utt in corpus.iter_utterances()]

print(f"Total utterances: {len(all_texts)}")
print("Example utterances:", all_texts[:5])

Total utterances: 304713
Example utterances: ['They do not!', 'They do to!', 'I hope so.', 'She okay?', "Let's go."]


N-grams for sentence uniqueness

In [31]:
#lets work with the movie conversations for sentence structure
tokenized_texts = [word_tokenize(text.lower()) for text in all_texts]


In [32]:
all_tokens = [token for sublist in tokenized_texts for token in sublist]

def calculate_uniqueness(expression, n):
    print(f'expression = {expression}')
    corpus_ngrams = list(ngrams(all_tokens, n))
    corpus_ngram_freq = FreqDist(corpus_ngrams)
    
    expr_tokens = word_tokenize(expression.lower())
    expr_ngrams = list(ngrams(expr_tokens, n))
    
    present_count = sum(1 for bg in expr_ngrams if bg in corpus_ngram_freq)
    proportion_present = present_count / len(expr_ngrams)
    print(f'Proportion of {n}grams present in corpus:', proportion_present)


In [33]:
print("A known example for bigrams")
calculate_uniqueness("That was a piece of cake!", 2)
print("A known example for trigrams")
calculate_uniqueness("That was a piece of cake!", 3)

A known example for bigrams
expression = That was a piece of cake!
Proportion of 2grams present in corpus: 1.0
A known example for trigrams
expression = That was a piece of cake!
Proportion of 3grams present in corpus: 0.8


In [34]:
print("An unknown example for bigrams")
calculate_uniqueness("de org bah monkey brawl banana blitz Hello There", 2)
print("An unknown example for trigrams")
calculate_uniqueness("de org bah monkey brawl banana blitz Hello There", 3)

An unknown example for bigrams
expression = de org bah monkey brawl banana blitz Hello There
Proportion of 2grams present in corpus: 0.125
An unknown example for trigrams
expression = de org bah monkey brawl banana blitz Hello There
Proportion of 3grams present in corpus: 0.0


Check against urban dictionary and oxford dictionary for single word expressions

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dfydata/the-online-plain-text-english-dictionary-opted")

print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

data_path = os.path.join(path, "OPTED-Dictionary.csv")

# Load the CSV
opted_df = pd.read_csv(
    data_path,
    sep=',',               # comma-separated
    engine='python',
    quotechar='"',
    header=0,              # first row is header
    on_bad_lines='skip',
    encoding='cp1252'
)

# Strip whitespace from all string columns
for col in opted_df.columns:
    opted_df[col] = opted_df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

# Build dictionary using lowercase words
oxford_dict = opted_df.groupby(opted_df['Word'].str.lower()).apply(
    lambda x: x[['Definition']].to_dict('records')
).to_dict()

100%|█████████████████████████████████████████████████████████████████████████████| 4.84M/4.84M [00:00<00:00, 20.1MB/s]

Extracting files...


Path to dataset files: C:\Users\brock\.cache\kagglehub\datasets\dfydata\the-online-plain-text-english-dictionary-opted\versions\1
C:\Users\brock\.cache\kagglehub\datasets\dfydata\the-online-plain-text-english-dictionary-opted\versions\1\OPTED-Dictionary.csv


In [8]:
def get_oxford_definition(word):
    word_lower = word.lower()
    if word_lower in oxford_dict:
            return True
    else:
        return False

# Test
get_oxford_definition("apple")

True

In [9]:
all_words = set(urban_dictionary['word'].str.lower())

def get_definition(word):
    definitions = urban_dictionary.loc[
        urban_dictionary['word'].str.lower() == word.lower(), 
        'definition'
    ]
    
    if definitions.empty:
        print("Word not found.")
    else:
        for d in definitions:
            print(d)

def check_dict(word):
    word = word.lower()
    if word in all_words:
        print("Word is not unique")
        print(word)
        print("Urban: ", end = ' ')
        get_definition(word)
    elif get_oxford_definition(word):
        print("Word is not unique")
        print(word)
        word_lower = word.lower()
        for entry in oxford_dict[word_lower]:
            print("Oxford:", entry['Definition'])
            break
    else:
        print(word)
        print("Word not found in either dictionary, Word is unique")

In [10]:
check_dict("Loppern")
print()
check_dict("table")
print()
check_dict("le")

loppern
Word not found in either dictionary, Word is unique

Word is not unique
table
Oxford: "A smooth  flat surface like the side of a board; a thin flat smooth piece of anything; a slab."

Word is not unique
le
Urban:  A new type of cancer that is prominently found in  comic  memes. It is the French equivalent of  the definite article  "the", however in certain memes, it is frivolously placed before any English noun. The exact origins of why "le" is used in memes is  nebulous .


LLM judge for semantic fluency, specfically gemma-3-12b-it


In [2]:
from transformers import pipeline
import torch

C:\Users\brock\miniconda3\envs\cs178\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0220 16:19:18.721000 20176 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
pipe = pipeline(
    "image-text-to-text",
    model="google/gemma-3-12b-it",
    torch_dtype=torch.bfloat16
)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  6.75it/s]
Device set to use cpu


In [12]:
import json
# Load the JSON file
with open("generated_slang.json", "r", encoding="utf-8") as f:
    lines = json.load(f)  # assumes a JSON array of strings

single_worded = []
multi_worded = []

for line in lines:
    if len(line.strip().split()) == 1:
        single_worded.append(line)
    else:
        multi_worded.append(line)

print("Single-word lines:", single_worded[:10])
print("Multi-word lines:", multi_worded[:10])

Single-word lines: ['sus', ' bussin', ' fr', ' slay', ' tea', ' yeet', ' cap', ' lowkey', ' mid', ' simp']
Multi-word lines: [' vibe check', ' no cap', ' main character', ' glow up', ' stan account', ' chaotic energy', ' fam vibes', ' bussin’ out', ' send tea', ' let’s get it']


In [15]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": f"Here are two lists of terms:\n"
            f"Single-worded: {single_worded}\n"
            f"Multi-worded: {multi_worded}\n\n"
            f"Rank each item in these two lists as being semantically fluent or not, only return the item followed by a yes or no, generate no additional text. Seperate each item score pair with a comma"
            f"For example sus:yes, bussin:no"}
        ]
    }
]

output = pipe(
    messages, 
    max_new_tokens=3000, 
    temperature=0.1, 
    top_p=0.9)
print(output[0]["generated_text"][-1]["content"])

sus:yes, bussin:no, fr:no, slay:yes, tea:yes, yeet:yes, cap:yes, lowkey:yes, mid:no, simp:yes, stan:yes, roast:yes, ghost:yes, drip:yes, flex:yes, clout:yes, bet:yes, slayage:yes, mood:yes, chaotic:yes, era:yes, vibe:yes, ting:no, fam:yes, bestie:yes, sis:yes, bro:yes, dude:yes, lit:yes, fire:yes, facts:yes, ghosting:yes, mid-tier:yes, friendzone:yes, vibe check:yes, no cap:yes, main character:yes, glow up:yes, stan account:yes, chaotic energy:yes, fam vibes:yes, bussin’ out:yes, send tea:yes, let’s get it:yes, "bet you didn't know":yes, that’s the tea:yes, real talk:yes, it’s giving:yes, serving looks:yes, slay queen:yes, king energy:yes, drip city:yes, main character syndrome:yes, chaotic good:yes, vibe squad:yes, tea spiller:yes, roast session:yes, simp alert:yes, stan life:yes, cap city:yes, lowkey obsessed:yes, glow up goals:yes, yeet yourself:yes, bussin’ meal:yes, tea time:yes, friend group:yes, besties forever:yes, bro code:yes, dude out:yes, fam jam:yes, vibe nation:yes, lit a

In [16]:
model_text = output[0]["generated_text"][-1]["content"]
pairs = [p.strip() for p in model_text.split(",") if p.strip()]

# Step 2: Count yes and no
counts = {"yes": 0, "no": 0}
for pair in pairs:
    if ":" in pair:
        item, score = pair.split(":", 1)
        score = score.strip().lower()
        if score in counts:
            counts[score] += 1

# Step 3: Calculate proportions
total = counts["yes"] + counts["no"]
proportions = {
    "yes": counts["yes"] / total if total > 0 else 0,
    "no": counts["no"] / total if total > 0 else 0
}

# Step 4: Combine data into JSON object
result_json = {
    "counts": counts,
    "proportions": proportions,
    "pairs": {p.split(":")[0].strip(): p.split(":")[1].strip() for p in pairs if ":" in p}
}

# Step 5: Export to a JSON file
with open("semantic_fluency.json", "w") as f:
    json.dump(result_json, f, indent=4)

# Optional: print the JSON
print(json.dumps(result_json, indent=4))

{
    "counts": {
        "yes": 168,
        "no": 4
    },
    "proportions": {
        "yes": 0.9767441860465116,
        "no": 0.023255813953488372
    },
    "pairs": {
        "sus": "yes",
        "bussin": "no",
        "fr": "no",
        "slay": "yes",
        "tea": "yes",
        "yeet": "yes",
        "cap": "yes",
        "lowkey": "yes",
        "mid": "no",
        "simp": "yes",
        "stan": "yes",
        "roast": "yes",
        "ghost": "yes",
        "drip": "yes",
        "flex": "yes",
        "clout": "yes",
        "bet": "yes",
        "slayage": "yes",
        "mood": "yes",
        "chaotic": "yes",
        "era": "yes",
        "vibe": "yes",
        "ting": "no",
        "fam": "yes",
        "bestie": "yes",
        "sis": "yes",
        "bro": "yes",
        "dude": "yes",
        "lit": "yes",
        "fire": "yes",
        "facts": "yes",
        "ghosting": "yes",
        "mid-tier": "yes",
        "friendzone": "yes",
        "vibe check": "yes",
 

In [39]:
for i in multi_worded:
    calculate_uniqueness(i, 2)

expression =  vibe check
Proportion of 2grams present in corpus: 0.0
expression =  no cap
Proportion of 2grams present in corpus: 1.0
expression =  main character
Proportion of 2grams present in corpus: 1.0
expression =  glow up
Proportion of 2grams present in corpus: 0.0
expression =  stan account
Proportion of 2grams present in corpus: 0.0
expression =  chaotic energy
Proportion of 2grams present in corpus: 0.0
expression =  fam vibes
Proportion of 2grams present in corpus: 0.0
expression =  bussin’ out
Proportion of 2grams present in corpus: 0.0
expression =  send tea
Proportion of 2grams present in corpus: 0.0
expression =  let’s get it
Proportion of 2grams present in corpus: 0.25
expression =  bet you didn't know
Proportion of 2grams present in corpus: 1.0
expression =  that’s the tea
Proportion of 2grams present in corpus: 0.5
expression =  real talk
Proportion of 2grams present in corpus: 1.0
expression =  it’s giving
Proportion of 2grams present in corpus: 0.0
expression =  ser

In [13]:
for i in single_worded:
    check_dict(i)

Word is not unique
sus
Urban:  when you   cant   spell  suspishos
 bussin
Word not found in either dictionary, Word is unique
 fr
Word not found in either dictionary, Word is unique
 slay
Word not found in either dictionary, Word is unique
 tea
Word not found in either dictionary, Word is unique
 yeet
Word not found in either dictionary, Word is unique
 cap
Word not found in either dictionary, Word is unique
 lowkey
Word not found in either dictionary, Word is unique
 mid
Word not found in either dictionary, Word is unique
 simp
Word not found in either dictionary, Word is unique
 stan
Word not found in either dictionary, Word is unique
 roast
Word not found in either dictionary, Word is unique
 ghost
Word not found in either dictionary, Word is unique
 drip
Word not found in either dictionary, Word is unique
 flex
Word not found in either dictionary, Word is unique
 clout
Word not found in either dictionary, Word is unique
 bet
Word not found in either dictionary, Word is unique
 slay

In [14]:
for i in multi_worded:
    check_dict(i)

 vibe check
Word not found in either dictionary, Word is unique
 no cap
Word not found in either dictionary, Word is unique
 main character
Word not found in either dictionary, Word is unique
 glow up
Word not found in either dictionary, Word is unique
 stan account
Word not found in either dictionary, Word is unique
 chaotic energy
Word not found in either dictionary, Word is unique
 fam vibes
Word not found in either dictionary, Word is unique
 bussin’ out
Word not found in either dictionary, Word is unique
 send tea
Word not found in either dictionary, Word is unique
 let’s get it
Word not found in either dictionary, Word is unique
 bet you didn't know
Word not found in either dictionary, Word is unique
 that’s the tea
Word not found in either dictionary, Word is unique
 real talk
Word not found in either dictionary, Word is unique
 it’s giving
Word not found in either dictionary, Word is unique
 serving looks
Word not found in either dictionary, Word is unique
 slay queen
Word not 